# 11_Model_Inference.ipynb

Load the trained multimodal model and predict Flood or Heatwave
for a single event from CNN + Transformer feature CSVs.


In [ ]:
!pip -q install torch pandas

import pandas as pd
import torch
import torch.nn as nn
from google.colab import files

print("Upload cnn_features.csv")
u=files.upload()
cnn=pd.read_csv(list(u.keys())[0])

print("Upload transformer_features.csv")
u=files.upload()
tr=pd.read_csv(list(u.keys())[0])

print("Upload multimodal_model.pth")
u=files.upload()
model_path=list(u.keys())[0]

def clean_ids(s):
    return (s.astype(str)
              .str.replace("_Sentinel-2","",regex=False)
              .str.replace("_Landsat","",regex=False)
              .str.strip())

cnn["event_id"]=clean_ids(cnn["event_id"])
tr["event_id"]=clean_ids(tr["event_id"])

cnn=cnn.rename(columns={c:f"cnn_{c}" for c in cnn.columns if c!="event_id"})
tr=tr.rename(columns={c:f"tr_{c}" for c in tr.columns if c!="event_id"})

df=cnn.merge(tr,on="event_id",how="inner")

print("\nAvailable Events:")
print(df["event_id"].tolist())

event_id=input("\nEnter event_id exactly as shown above: ").strip()

row=df[df["event_id"]==event_id]

if len(row)==0:
    raise ValueError("Event ID not found.")

cnn_cols=[c for c in df.columns if c.startswith("cnn_")]
tr_cols=[c for c in df.columns if c.startswith("tr_")]

cnn_tensor=torch.tensor(row[cnn_cols].values,dtype=torch.float32)
tr_tensor=torch.tensor(row[tr_cols].values,dtype=torch.float32)

class FusionModel(nn.Module):
    def __init__(self,dim=512):
        super().__init__()
        self.attn=nn.MultiheadAttention(dim,8,batch_first=True)
        self.head=nn.Sequential(
            nn.Linear(dim,256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256,2)
        )
    def forward(self,cnn,tr):
        out,_=self.attn(cnn.unsqueeze(1),tr.unsqueeze(1),tr.unsqueeze(1))
        return self.head(out.squeeze(1))

device="cuda" if torch.cuda.is_available() else "cpu"
model=FusionModel().to(device)
model.load_state_dict(torch.load(model_path,map_location=device))
model.eval()

with torch.no_grad():
    logits=model(cnn_tensor.to(device),tr_tensor.to(device))
    probs=torch.softmax(logits,dim=1).cpu().numpy()[0]

classes=["FLASH","HEAT"]
pred_idx=probs.argmax()

print("\n===== Prediction =====")
print("Event ID:",event_id)
print("Predicted Class:",classes[pred_idx])
print(f"FLASH Probability : {probs[0]*100:.2f}%")
print(f"HEAT Probability  : {probs[1]*100:.2f}%")
